# Estadísticas Descriptivas - Congressional Trading

Versión mejorada con estilo unificado para paper.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================
# PATHS
# ============================================
BASE_PATH = r'C:\Users\sebib\Documents\GitHub\US_Congress'
DATA_PATH = os.path.join(BASE_PATH, 'data', 'congress_trades', 'congress_trades_with_committees.parquet')
FIGURES_PATH = os.path.join(BASE_PATH, 'figures')
os.makedirs(FIGURES_PATH, exist_ok=True)

# ============================================
# PALETA DE COLORES UNIFICADA
# ============================================
COLORS = {
    'buy': '#27ae60',       # Verde
    'sell': '#c0392b',      # Rojo
    'democrat': '#2980b9',  # Azul
    'republican': '#c0392b', # Rojo
    'independent': '#7f8c8d', # Gris
    'house': '#3498db',     # Azul claro
    'senate': '#9b59b6',    # Púrpura
    'neutral': '#34495e',   # Gris oscuro
    'accent': '#f39c12'     # Naranja
}

# ============================================
# ESTILO PARA PAPER
# ============================================
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.size': 11,
    'font.family': 'sans-serif',
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--'
})

# Cargar datos
df = pd.read_parquet(DATA_PATH)
print(f"Observaciones: {len(df):,}")
print(f"Columnas: {len(df.columns)}")

In [ ]:
# Preparar variables
df['trade_date'] = pd.to_datetime(df['trade_date'])
df['trade_year'] = df['trade_date'].dt.year
df['trade_month'] = df['trade_date'].dt.to_period('M')

# Identificar compras y ventas
df['is_buy'] = df['Transaction'].str.lower().str.contains('purchase|buy', na=False).astype(int)
df['is_sell'] = df['Transaction'].str.lower().str.contains('sale|sell', na=False).astype(int)

# Limpiar Party para que sea consistente
df['Party_clean'] = df['Party'].replace({'D': 'Democrat', 'R': 'Republican', 'I': 'Independent'})

print(f"Período: {df['trade_date'].min().date()} a {df['trade_date'].max().date()}")
print(f"\nPartidos: {df['Party_clean'].value_counts().to_dict()}")

---
## STAT 1: Trades por Año

In [ ]:
# Datos
trades_by_year = df.groupby('trade_year').agg(
    Compras=('is_buy', 'sum'),
    Ventas=('is_sell', 'sum')
).reset_index()

# Gráfico
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(trades_by_year))
width = 0.4

bars1 = ax.bar(x - width/2, trades_by_year['Compras'], width, 
               label='Compras', color=COLORS['buy'], edgecolor='white', linewidth=0.8)
bars2 = ax.bar(x + width/2, trades_by_year['Ventas'], width, 
               label='Ventas', color=COLORS['sell'], edgecolor='white', linewidth=0.8)

ax.set_xlabel('Año')
ax.set_ylabel('Número de Transacciones')
ax.set_title('Transacciones de Congresistas por Año')
ax.set_xticks(x)
ax.set_xticklabels(trades_by_year['trade_year'], rotation=45, ha='right')
ax.legend(frameon=True, fancybox=True, shadow=False)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'stat1.png'), dpi=300, bbox_inches='tight', facecolor='white')
print(f"Guardado: stat1.png")
plt.show()

---
## STAT 2: Trades por Cámara

In [ ]:
# Datos
trades_by_chamber = df.groupby('Chamber').agg(
    Total=('is_buy', 'count'),
    Compras=('is_buy', 'sum'),
    Ventas=('is_sell', 'sum')
).reset_index()

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
colors_pie = [COLORS['house'], COLORS['senate']]
wedges, texts, autotexts = axes[0].pie(
    trades_by_chamber['Total'], 
    labels=trades_by_chamber['Chamber'], 
    autopct='%1.1f%%',
    colors=colors_pie, 
    explode=(0.02, 0.02), 
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for autotext in autotexts:
    autotext.set_fontsize(11)
    autotext.set_fontweight('bold')
axes[0].set_title('Distribución de Trades por Cámara')

# Bar chart
x = np.arange(len(trades_by_chamber))
width = 0.35
axes[1].bar(x - width/2, trades_by_chamber['Compras'], width, 
            label='Compras', color=COLORS['buy'], edgecolor='white')
axes[1].bar(x + width/2, trades_by_chamber['Ventas'], width, 
            label='Ventas', color=COLORS['sell'], edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(trades_by_chamber['Chamber'])
axes[1].set_ylabel('Número de Transacciones')
axes[1].set_title('Compras vs Ventas por Cámara')
axes[1].legend(frameon=True)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'stat2.png'), dpi=300, bbox_inches='tight', facecolor='white')
print(f"Guardado: stat2.png")
plt.show()

---
## STAT 3: Trades por Partido

In [ ]:
# Datos
trades_by_party = df.groupby('Party_clean').agg(
    Total=('is_buy', 'count')
).reset_index().sort_values('Total', ascending=True)

# Mapear colores correctamente
color_map = {
    'Democrat': COLORS['democrat'],
    'Republican': COLORS['republican'],
    'Independent': COLORS['independent']
}
bar_colors = [color_map.get(p, COLORS['neutral']) for p in trades_by_party['Party_clean']]

# Gráfico
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(trades_by_party['Party_clean'], trades_by_party['Total'], 
               color=bar_colors, edgecolor='white', linewidth=0.8, height=0.6)

ax.set_xlabel('Número de Transacciones')
ax.set_title('Transacciones por Partido Político')

# Agregar valores al final de cada barra
for bar, val in zip(bars, trades_by_party['Total']):
    ax.text(val + 500, bar.get_y() + bar.get_height()/2, f'{val:,}', 
            va='center', fontsize=10, fontweight='bold')

ax.set_xlim(0, trades_by_party['Total'].max() * 1.15)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'stat3.png'), dpi=300, bbox_inches='tight', facecolor='white')
print(f"Guardado: stat3.png")
plt.show()

---
## STAT 4: Top 10 Tickers

In [ ]:
# Datos
top_tickers = df.groupby('Ticker_Clean').agg(
    Total=('is_buy', 'count'),
    Compras=('is_buy', 'sum'),
    Ventas=('is_sell', 'sum')
).reset_index().sort_values('Total', ascending=False).head(10)

# Gráfico
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(top_tickers))
width = 0.4

bars1 = ax.bar(x - width/2, top_tickers['Compras'], width, 
               label='Compras', color=COLORS['buy'], edgecolor='white', linewidth=0.8)
bars2 = ax.bar(x + width/2, top_tickers['Ventas'], width, 
               label='Ventas', color=COLORS['sell'], edgecolor='white', linewidth=0.8)

ax.set_xlabel('Ticker')
ax.set_ylabel('Número de Transacciones')
ax.set_title('Top 10 Acciones Más Operadas por Congresistas')
ax.set_xticks(x)
ax.set_xticklabels(top_tickers['Ticker_Clean'], rotation=45, ha='right', fontweight='bold')
ax.legend(frameon=True)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'stat4.png'), dpi=300, bbox_inches='tight', facecolor='white')
print(f"Guardado: stat4.png")
plt.show()

---
## STAT 5: Top 10 Congresistas

In [ ]:
# Datos
top_politicians = df.groupby(['Name', 'Party_clean', 'Chamber']).agg(
    Total=('is_buy', 'count')
).reset_index().sort_values('Total', ascending=False).head(10)

# Invertir para que el mayor esté arriba
top_politicians = top_politicians.iloc[::-1]

# Mapear colores por partido
color_map = {
    'Democrat': COLORS['democrat'],
    'Republican': COLORS['republican'],
    'Independent': COLORS['independent']
}
bar_colors = [color_map.get(p, COLORS['neutral']) for p in top_politicians['Party_clean']]

# Crear labels con nombre y cámara
labels = [f"{name} ({chamber[0]})" for name, chamber in zip(top_politicians['Name'], top_politicians['Chamber'])]

# Gráfico
fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.barh(labels, top_politicians['Total'], color=bar_colors, 
               edgecolor='white', linewidth=0.8, height=0.7)

ax.set_xlabel('Número de Transacciones')
ax.set_title('Top 10 Congresistas con Más Transacciones')

# Agregar valores
for bar, val in zip(bars, top_politicians['Total']):
    ax.text(val + 200, bar.get_y() + bar.get_height()/2, f'{val:,}', 
            va='center', fontsize=10, fontweight='bold')

# Leyenda
legend_elements = [
    mpatches.Patch(facecolor=COLORS['democrat'], edgecolor='white', label='Democrat'),
    mpatches.Patch(facecolor=COLORS['republican'], edgecolor='white', label='Republican')
]
ax.legend(handles=legend_elements, loc='lower right', frameon=True)

ax.set_xlim(0, top_politicians['Total'].max() * 1.15)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'stat5.png'), dpi=300, bbox_inches='tight', facecolor='white')
print(f"Guardado: stat5.png")
plt.show()

---
## STAT 6: Distribución de Montos

In [ ]:
# Datos
amount_dist = df['Trade_Size_USD'].value_counts().reset_index()
amount_dist.columns = ['Rango', 'Cantidad']
amount_dist['%'] = (amount_dist['Cantidad'] / amount_dist['Cantidad'].sum() * 100).round(1)

# Ordenar
order = ['$1,001 - $15,000', '$15,001 - $50,000', '$50,001 - $100,000', 
         '$100,001 - $250,000', '$250,001 - $500,000', '$500,001 - $1,000,000',
         '$1,000,001 - $5,000,000', 'Over $5,000,000']
amount_dist['Rango'] = pd.Categorical(amount_dist['Rango'], categories=order, ordered=True)
amount_dist = amount_dist.sort_values('Rango').dropna()

# Etiquetas más cortas
short_labels = ['$1K-15K', '$15K-50K', '$50K-100K', '$100K-250K', 
                '$250K-500K', '$500K-1M', '$1M-5M', '>$5M']

# Gráfico
fig, ax = plt.subplots(figsize=(12, 6))

# Gradiente de colores
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(amount_dist)))

bars = ax.bar(range(len(amount_dist)), amount_dist['Cantidad'], 
              color=colors, edgecolor='white', linewidth=0.8)

ax.set_xticks(range(len(amount_dist)))
ax.set_xticklabels(short_labels[:len(amount_dist)], rotation=45, ha='right')
ax.set_ylabel('Número de Transacciones')
ax.set_xlabel('Rango de Monto')
ax.set_title('Distribución de Transacciones por Rango de Monto')

# Agregar porcentajes
for bar, pct in zip(bars, amount_dist['%']):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 500, f'{pct}%', 
            ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'stat6.png'), dpi=300, bbox_inches='tight', facecolor='white')
print(f"Guardado: stat6.png")
plt.show()

---
## STAT 7: Disclosure Delay

In [ ]:
# Calcular delay
if 'Filed' in df.columns:
    df['filed_date'] = pd.to_datetime(df['Filed'])
    df['disclosure_delay'] = (df['filed_date'] - df['trade_date']).dt.days
    df['disclosure_delay'] = df['disclosure_delay'].clip(lower=0, upper=365)

# Gráfico
fig, ax = plt.subplots(figsize=(12, 6))

n, bins, patches = ax.hist(df['disclosure_delay'].dropna(), bins=50, 
                            color=COLORS['house'], edgecolor='white', linewidth=0.5, alpha=0.85)

# Líneas de referencia
median_delay = df['disclosure_delay'].median()
ax.axvline(x=30, color=COLORS['sell'], linestyle='--', linewidth=2.5, label='Límite 30 días')
ax.axvline(x=45, color=COLORS['accent'], linestyle='--', linewidth=2.5, label='Límite 45 días')
ax.axvline(x=median_delay, color=COLORS['buy'], linestyle='-', linewidth=2.5, 
           label=f'Mediana ({median_delay:.0f} días)')

ax.set_xlabel('Días entre Transacción y Divulgación')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución del Rezago de Divulgación')
ax.legend(frameon=True, loc='upper right')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'stat7.png'), dpi=300, bbox_inches='tight', facecolor='white')
print(f"Guardado: stat7.png")
plt.show()

---
## STAT 8: Evolución Temporal

In [ ]:
# Datos mensuales
monthly_trades = df.groupby('trade_month').agg(
    Compras=('is_buy', 'sum'),
    Ventas=('is_sell', 'sum')
).reset_index()
monthly_trades['trade_month'] = monthly_trades['trade_month'].dt.to_timestamp()

# Gráfico
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(monthly_trades['trade_month'], monthly_trades['Compras'], 
        label='Compras', color=COLORS['buy'], linewidth=1.8, alpha=0.9)
ax.plot(monthly_trades['trade_month'], monthly_trades['Ventas'], 
        label='Ventas', color=COLORS['sell'], linewidth=1.8, alpha=0.9)

# Evento COVID
ax.axvline(x=pd.Timestamp('2020-03-01'), color=COLORS['neutral'], 
           linestyle='--', linewidth=2, alpha=0.7, label='COVID-19 (Mar 2020)')

ax.set_xlabel('Fecha')
ax.set_ylabel('Número de Transacciones')
ax.set_title('Evolución Mensual de Transacciones de Congresistas')
ax.legend(frameon=True, loc='upper left')

# Formato de fechas
import matplotlib.dates as mdates
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'stat8.png'), dpi=300, bbox_inches='tight', facecolor='white')
print(f"Guardado: stat8.png")
plt.show()

---
## RESUMEN

In [ ]:
print("\n" + "="*60)
print("FIGURAS GENERADAS")
print("="*60)
print(f"\nCarpeta: {FIGURES_PATH}")
print("\n  stat1.png : Trades por año")
print("  stat2.png : Trades por cámara")
print("  stat3.png : Trades por partido")
print("  stat4.png : Top 10 tickers")
print("  stat5.png : Top 10 congresistas")
print("  stat6.png : Distribución de montos")
print("  stat7.png : Disclosure delay")
print("  stat8.png : Evolución temporal")
print("\n" + "="*60)